# Vase Selection — 1D CNN Classifier

## Goal
Predict which vase a participant will select, given the shape of the vase as a profile curve.

Each vase is defined by a sequence of `(x, y)` coordinate pairs that trace its silhouette. Participants choose between **two** vases per trial across 5 rounds of 5 generations. The model learns which profile shapes tend to be selected across participants, then generalises to new participants it has never seen.

## What the data actually looks like
Facts verified against `xy_data.parquet` — these drive several design choices below:

| Property | Value |
|---|---|
| Rows | 34,050,000 (one per sampled contour point) |
| Trajectories | 136,200 = 2,724 participants × 25 trials × 2 vases |
| Points per trajectory | **exactly 250**, always — no padding needed |
| Vases per trial | **exactly 2**, with **exactly 1** selected |
| Base rate | **exactly 50.0%** selected |
| Contour | a **closed loop**, traced at uniform arc length (step ≈ 0.045, first↔last gap 0.039) |
| Sessions | 2,724 — strictly 1:1 with participants, no duplicate ingestion |

## Two consequences worth stating up front

**1. The contour is not a function `y(x)`.** `x` spans −4.9 → +4.9 and doubles back on itself; file order is the traversal order around the silhouette. Sorting points by `x` scrambles the curve (mean gap between neighbouring points jumps from 0.045 to 1.36) and leaves the CNN convolving over noise. We preserve file order.

**2. The base rate is exactly 50%, so per-vase accuracy at a 0.5 threshold is chance.** A model that has learned nothing prints `acc 0.500`. This notebook therefore reports **pairwise accuracy** — score both vases in a trial, pick the higher — which is the actual task metric, alongside ROC-AUC. Note this is an *evaluation* change only; the model is still trained on vases independently.

## Pipeline Overview
| Step | What happens |
|---|---|
| **1. Load data** | Read the raw parquet — one row per `(x, y)` sample point |
| **2. Build trajectories** | Group rows into per-participant, per-vase closed contours, preserving traversal order |
| **3. Assemble arrays** | Stack to `(n, 2, 250)` for `Conv1d` |
| **4. Split & normalise** | Three-way split by `participant_id`; normalisation stats from **train only** |
| **5. Baselines** | Chance (50%) and the `is_parent_vase` heuristic the CNN must beat |
| **6. Model** | 1D CNN with **circular** padding (the contour is a loop) → one logit per vase |
| **7. Train** | BCE loss, Adam, LR schedule and early model selection on **validation**, never test |
| **8. Evaluate** | Held-out test set scored once, at the end |

In [ ]:
import copy
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss, confusion_matrix

from tqdm import tqdm

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Apple Silicon: fall through to MPS before CPU. The first run of this notebook
# took ~90-110 s/epoch on CPU (and crept upward, i.e. thermal throttling).
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Using device:", device)

FIG_DIR = Path("figures/vase_select_cnn")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# House palette — same values as density_diff/pc_density_diff_map.ipynb
BLUE, ORANGE, RED = "#2a78d6", "#eb6834", "#e34948"
INK, MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
GRAY_MARK = "#c3c2b7"

In [ ]:
COLS = [
    "participant_id", "generation_key", "vase",
    "x", "y", "is_selected_vase", "is_parent_vase",
]
xy_dataset = pd.read_parquet(
    "../file_processing/processed/xy_data.parquet", columns=COLS
)
print("Dataset shape:", xy_dataset.shape)
xy_dataset.head()

## Step 2 — Build Trajectories

### What this does
The raw dataset has one row per sampled point on a vase contour. We group rows by `(participant_id, generation_key, vase)` to reconstruct each vase's full outline as seen by each participant.

Each group becomes one **trajectory** — a 2D array of shape `(250, 2)` where the columns are the `x` and `y` coordinates of a point on the silhouette.

### Why we do *not* sort by `x`
The outline is a **closed contour** traced at uniform arc length, not a height profile. Consecutive rows in the file are consecutive points as you walk around the silhouette, ~0.045 apart, closing back on the start. `x` runs −4.9 → +4.9 and is non-monotonic in every single trajectory.

Sorting by `x` would:
- turn the `x` channel into a meaningless monotone ramp,
- shuffle the `y` channel into an order with no spatial structure (neighbouring points end up ~1.36 apart on average, 30× the true spacing),
- leave `Conv1d` — which only ever looks at *adjacent* positions — with nothing real to convolve over.

So we sort only by the grouping keys, with a **stable** sort that preserves the original within-group row order. An assertion checks the resulting contours really are uniformly spaced.

### Why include `participant_id` in the groupby
The same vase shape is only ever shown to one participant (verified: zero shapes are shared across participants), but including `participant_id` still:
- makes each trajectory one sample (one participant × one vase × one generation),
- lets us split cleanly by participant later (see Step 4).

### Outputs
| Variable | Description |
|---|---|
| `sequences` | List of `(250, 2)` numpy arrays — one per trajectory |
| `labels` | List of ints — `1` if this participant selected this vase, `0` otherwise |
| `participant_ids` | Parallel array of participant ID strings, used for the split |
| `trial_ids` | Parallel array of `"{pid}|{generation_key}"` — the two vases of a trial share one | 
| `is_parent` | Parallel bool array — was this vase the previous generation's winner |

In [ ]:
# NOTE: sort by the grouping keys ONLY, stably, so the within-group traversal
# order of the closed contour survives. Do NOT add "x" to this key list.
nn_dataset = xy_dataset.sort_values(
    ["participant_id", "generation_key", "vase"], kind="stable"
)

sequences, labels, participant_ids, trial_ids, is_parent = [], [], [], [], []
for (pid, gen_key, vase), group in nn_dataset.groupby(
    ["participant_id", "generation_key", "vase"], sort=False
):
    sequences.append(group[["x", "y"]].values)  # (250, 2), traversal order
    labels.append(int(group["is_selected_vase"].iloc[0]))
    participant_ids.append(pid)
    trial_ids.append(f"{pid}|{gen_key}")
    is_parent.append(bool(group["is_parent_vase"].iloc[0]))

    # labels/parent flags must be constant within a trajectory
    assert group["is_selected_vase"].nunique() == 1
    assert group["is_parent_vase"].nunique() == 1

participant_ids = np.array(participant_ids)
trial_ids = np.array(trial_ids)
is_parent = np.array(is_parent)
labels = np.array(labels)
lengths = [len(s) for s in sequences]

# Sanity check: contours must be uniformly sampled. If someone re-introduces a
# sort by x, mean_step blows up by ~30x and this fires.
_steps = np.concatenate([
    np.linalg.norm(np.diff(s, axis=0), axis=1) for s in sequences[:200]
])
mean_step, max_step = _steps.mean(), _steps.max()
assert max_step < 5 * mean_step, (
    f"Contour ordering looks scrambled (mean step {mean_step:.3f}, "
    f"max {max_step:.3f}) — is the data being sorted by x?"
)

print(f"Trajectories: {len(sequences)}")
print(f"Unique participants: {len(np.unique(participant_ids))}")
print(f"Unique trials: {len(np.unique(trial_ids))}")
print(f"Sequence length — min: {min(lengths)}, max: {max(lengths)}, median: {int(np.median(lengths))}")
print(f"Selected: {labels.sum()} / {len(labels)} ({100*labels.mean():.1f}%)")
print(f"Contour step — mean {mean_step:.4f}, max {max_step:.4f}  (uniform arc length OK)")

## Step 3 — Assemble Arrays

### No padding needed
Every trajectory is **exactly 250 points**, so the padding logic that used to live here was dead code. We assert the invariant instead.

### The outline is a half-profile, reflected

The vases are generated as a half-profile mirrored about `x = 0`, so **the 250-point contour traces the same curve twice**. Reflection maps index `i → (1 − i) mod 250`, pairing the samples into 125 mirror pairs: `(0,1), (2,249), (3,248), … (125,126)`.

Measured on the data: the median per-point residual `|xᵢ + x_mirror(i)|` is **0.0075**, which is **0.24% of vase width** and well under one sampling step of 0.0465. The residual is resampling and float noise, not real asymmetry — the cell asserts this rather than assuming it.

So the contour carries **125 points of independent information, not 250**. Feeding both halves makes the model spend capacity learning that two identical copies are identical, and doubles the compute for zero information.

`HALF_CONTOUR = True` therefore slices indices **1–125** — one representative of each mirror pair, forming a contiguous arc from the rim down to the foot — and that is what the model sees. `X_full` keeps the whole closed contour, because every figure draws whole vases.

Set `HALF_CONTOUR = False` to feed the full loop and reproduce the earlier result.

### Why this also fixes the padding mode
The half-profile is an **open** curve whose endpoints sit on the symmetry axis, so its true continuation past each end is its own reflection → `padding_mode="reflect"`. The full closed loop wraps → `"circular"`. `PAD_MODE` is set from `HALF_CONTOUR`, so the padding follows the geometry rather than being a judgement call.

### What was removed
An earlier `DROP_DUPLICATE_SHAPES` flag lived here and was **unsound**: from generation 2 onward every trial is (parent, novel child) and the parent *is* the repeat, so dropping repeated shapes left 1-vase trials and broke every pairwise metric. There is no pair-safe vase-level dedup. The pair-safe alternative is to keep whole trials and restrict to generation 1, where both vases are freshly sampled and every shape is distinct — at the cost of 80% of the data. Deduplication for *display* happens where it belongs, in the Step 9 ranking figure.

### Transposing to `(n, 2, L)`
`Conv1d` expects `(batch, channels, length)`. Channels = 2: the `x` and `y` coordinates, like R/G/B in an image CNN.

### Normalisation happens in Step 4
Stats are fit on the **training split only** — the earlier version leaked the test set into preprocessing — and use a per-channel centre with a **single shared scale**, which preserves each vase's aspect ratio. Scaling `x` and `y` independently would stretch every silhouette by a different amount per axis, blurring exactly the distinction the model must learn.

### Repeated shapes
~40% of trajectories are exact duplicates: the vase selected at generation *N* reappears as the parent at *N+1*. Duplicates never cross participants, so this is **not** train/test leakage, but the effective sample size is smaller than it looks and repetition correlates with the label.


In [ ]:
import hashlib

FULL_LEN = 250
assert set(lengths) == {FULL_LEN}, f"Expected uniform length {FULL_LEN}, got {set(lengths)}"

# Full closed contour. Kept for PLOTTING — every figure draws whole vases —
# and as the source the model input is sliced from.
X_full = np.stack(sequences).transpose(0, 2, 1).astype(np.float32)
X_raw = X_full                       # alias: the figure cells refer to X_raw
y = labels.astype(np.float32)

shape_hash = np.array([
    hashlib.md5(np.ascontiguousarray(np.round(s, 6)).tobytes()).hexdigest()
    for s in sequences
])
gen_num = np.array([int(t.split("_")[-1]) for t in trial_ids])

n_unique = len(np.unique(shape_hash))
print(f"Distinct shapes: {n_unique} of {len(shape_hash)} "
      f"({100*(1 - n_unique/len(shape_hash)):.1f}% repeats)")

# --- Mirror symmetry -------------------------------------------------------
# The vases are generated as a half-profile reflected about x = 0, so the
# outline traces the SAME curve twice. Reflection maps index i -> (1 - i) mod 250,
# which pairs the 250 samples into 125 mirror pairs: (0,1), (2,249), ... (125,126).
MIRROR = (1 - np.arange(FULL_LEN)) % FULL_LEN

_s = X_full[:2000]
_dev = np.abs(_s[:, 0, :] + _s[:, 0, MIRROR])
_width = np.ptp(_s[:, 0, :], axis=1, keepdims=True)
print(f"\nMirror residual: median {np.median(_dev):.4f} "
      f"= {np.median(_dev / _width):.2%} of vase width, "
      f"vs a sampling step of {mean_step:.4f}")
assert np.median(_dev) < mean_step, (
    "contours are not mirror-symmetric to within one sampling step — "
    "HALF_CONTOUR would discard real information"
)
print("-> residual is sub-sample-spacing, i.e. the halves are the same curve.")

# --- Model input: one half only -------------------------------------------
# Indices 1..125 are one representative of each mirror pair, and they form a
# contiguous arc from the rim down to the foot. Feeding both halves would make
# the model spend capacity learning that two identical copies are identical.
HALF = slice(1, 126)
HALF_CONTOUR = True

X_model = np.ascontiguousarray(X_full[:, :, HALF]) if HALF_CONTOUR else X_full
SEQ_LEN = X_model.shape[2]

# The half-profile is an OPEN curve whose endpoints sit on the symmetry axis, so
# its true continuation past each end is its own reflection -> "reflect".
# The full closed loop wraps instead -> "circular". This is geometry, not taste.
PAD_MODE = "reflect" if HALF_CONTOUR else "circular"

print(f"\nModel input: {X_model.shape}  SEQ_LEN={SEQ_LEN}  padding_mode='{PAD_MODE}'")
print(f"X_full (for plots): {X_full.shape}   y: {y.shape} "
      f"(positive rate {y.mean():.3f})")


## Step 4 — Split & Normalise

### Why split by participant
If we split trajectories randomly, the same participant would appear in both train and test, and the model could memorise participant-specific preferences rather than learning generalisable shape features. Splitting on `participant_id` guarantees the test set contains only unseen participants.

It also guarantees something the pairwise metric needs: because `trial_id` is prefixed by `participant_id`, **both vases of a trial always land in the same split**. We can score a complete pair without reaching across the boundary.

### Why three splits, not two
The original notebook fed test loss to `ReduceLROnPlateau` and invited you to pick the epoch count by watching test loss. Both are model selection, and doing them on test turns your "honest estimate of real-world performance" into an optimistically biased one.

So: **train** fits weights, **validation** drives the LR schedule and picks the best checkpoint, **test** is touched exactly once, in Step 8.

| Split | Share of participants |
|---|---|
| Train | 70% |
| Validation | 15% |
| Test | 15% |

### Normalisation, fit on train only
```
mu     = per-channel mean over training trajectories   → shape (2,)
scale  = ONE std over both channels of training data   → scalar
X      = (X_raw - mu) / scale
```
Centring per channel is right (`x` and `y` have different natural offsets). Scaling by a single shared number is what preserves each vase's aspect ratio — see Step 3. `mu` and `scale` are kept in `NORM_STATS` so inference on new data reuses them rather than recomputing.

### Parameters
| Parameter | Default | Effect |
|---|---|---|
| `test_size` | 0.30 then 0.50 | 70/15/15 participant split |
| `random_state=SEED` | Fixed seed | Change for a different split; average across several seeds for a robust estimate |
| `batch_size=64` | 64 trajectories per batch | Larger → faster but more memory; smaller → noisier gradients, mild regularisation |

In [ ]:
class VaseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# --- Three-way split on participant IDs (70 / 15 / 15) ---------------------
unique_pids = np.unique(participant_ids)
train_pids, holdout_pids = train_test_split(
    unique_pids, test_size=0.30, random_state=SEED
)
val_pids, test_pids = train_test_split(
    holdout_pids, test_size=0.50, random_state=SEED
)

train_mask = np.isin(participant_ids, train_pids)
val_mask   = np.isin(participant_ids, val_pids)
test_mask  = np.isin(participant_ids, test_pids)

assert not (train_mask & val_mask).any()
assert not (train_mask & test_mask).any()
assert not (val_mask & test_mask).any()
assert (train_mask | val_mask | test_mask).all()
# both vases of a trial must stay together, or pairwise scoring is meaningless
for m in (train_mask, val_mask, test_mask):
    assert pd.Series(trial_ids[m]).value_counts().eq(2).all(), "split broke a trial pair"

# --- Normalise: stats from TRAIN ONLY, single shared scale ------------------
mu = X_model[train_mask].mean(axis=(0, 2)).astype(np.float32)   # (2,) per-channel centre
scale = np.float32(X_model[train_mask].std())                    # scalar, preserves aspect ratio
NORM_STATS = {"mu": mu, "scale": scale}   # reuse these at inference time

X = (X_model - mu.reshape(1, 2, 1)) / (scale + 1e-8)
print(f"Normalisation — mu {mu}, shared scale {scale:.4f}")

# --- Loaders ---------------------------------------------------------------
BATCH_SIZE = 64
train_ds = VaseDataset(X[train_mask], y[train_mask])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
# same tensors, unshuffled, for metrics: eval order must match trial_ids[train_mask]
train_eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False)

val_loader  = DataLoader(VaseDataset(X[val_mask],  y[val_mask]),  batch_size=256, shuffle=False)
test_loader = DataLoader(VaseDataset(X[test_mask], y[test_mask]), batch_size=256, shuffle=False)

for name, m, pids in [
    ("Train", train_mask, train_pids),
    ("Val  ", val_mask,   val_pids),
    ("Test ", test_mask,  test_pids),
]:
    print(f"{name}: {m.sum():6d} trajectories | {len(pids):5d} participants "
          f"| {m.sum()//2:5d} trials | positive rate {y[m].mean():.3f}")

## Step 5 — Baselines

Before training anything, establish what "doing nothing" scores. Two numbers matter.

### Chance = 50.0%, exactly
Every trial is two vases with exactly one selected. So the positive rate is exactly 0.5 and **per-vase accuracy at a 0.5 threshold is chance**. A model that has learned nothing still prints `acc 0.500`, which reads like it is nearly working. This is the single most misleading number the original notebook produced.

The metric that actually reflects the task is **pairwise accuracy**: score both vases in a trial, predict the higher-scoring one, check whether it was selected. Chance is still 50%, but unlike thresholded accuracy it moves as soon as the model learns anything, and it cannot be gamed by a shifted decision threshold. We also report **ROC-AUC**, which is threshold-free.

### The `is_parent_vase` heuristic ≈ 60%
From generation 2 onward, one of the two vases on screen is the participant's own winner from the previous generation. They re-pick it **62.2%** of the time (64.5% at gen 2, decaying to 59.4% at gen 5). Guessing "whichever is the parent", and flipping a coin on generation-1 trials where neither is, gives roughly **60% pairwise accuracy from a single boolean**.

The CNN never sees `is_parent_vase` — it only sees geometry — so this is not a feature it can copy. But it is the number a shape-only model has to be judged against. A CNN at 55% pairwise is learning something real about shape while still being worse than a one-line heuristic, and reporting it against 50% would hide that.

### Pair layout
Trajectories were built sorted by `(participant_id, generation_key, vase)`, so the two vases of a trial are **adjacent rows**. That lets every pairwise metric be a `.reshape(-1, 2)` rather than a groupby. The helper below asserts the layout holds rather than trusting it.

In [ ]:
def as_pairs(arr, mask):
    """Rows of `arr` for `mask`, viewed as (n_trials, 2) — the two vases of a trial."""
    out = np.asarray(arr)[mask]
    assert len(out) % 2 == 0, "odd number of trajectories in split"
    return out.reshape(-1, 2)


# the two rows of each pair must belong to the same trial, and exactly one be selected
for m in (train_mask, val_mask, test_mask):
    tp = as_pairs(trial_ids, m)
    assert (tp[:, 0] == tp[:, 1]).all(), "pairs are not adjacent — check the Step 2 sort"
    assert (as_pairs(y, m).sum(axis=1) == 1).all(), "a trial does not have exactly 1 winner"


def pairwise_correct(scores, mask):
    """Per-trial score in {0, 0.5, 1}: was the higher-scored vase the selected one?
    A tie earns 0.5 — what a coin flip would."""
    s = np.asarray(scores).reshape(-1, 2)
    lab = as_pairs(y, mask)
    hit = (s.argmax(axis=1) == lab.argmax(axis=1)).astype(float)
    return np.where(s[:, 0] == s[:, 1], 0.5, hit)


def pairwise_accuracy(scores, mask):
    """Fraction of trials where the higher-scored vase is the selected one."""
    return float(pairwise_correct(scores, mask).mean())


# Generation number per trial (1-5), for the per-generation breakdown in Step 9.
# trial_id is "{pid}|round_R_gen_G", so the trailing token is the generation.
gen_of_trial = {
    m_name: np.array([int(t.split("_")[-1]) for t in as_pairs(trial_ids, m)[:, 0]])
    for m_name, m in [("train", train_mask), ("val", val_mask), ("test", test_mask)]
}
# Participant per trial — needed for the cluster bootstrap (trials within a
# participant are correlated, so resampling trials would understate the CIs).
pid_of_trial = {
    m_name: as_pairs(participant_ids, m)[:, 0]
    for m_name, m in [("train", train_mask), ("val", val_mask), ("test", test_mask)]
}


# --- Baseline 1: chance ----------------------------------------------------
print(f"Chance pairwise accuracy: {0.5:.4f}\n")

# --- Baseline 2: pick the parent vase --------------------------------------
def parent_scores(mask):
    """Score 1 for the parent vase, 0 otherwise. At gen 1 neither is a parent,
    so both score 0 -> tie -> 0.5, i.e. a coin flip."""
    return as_pairs(is_parent, mask).astype(float).reshape(-1)


def parent_baseline(mask, label):
    par = as_pairs(is_parent, mask).astype(float)
    has_parent = par.sum(axis=1) == 1
    acc = pairwise_accuracy(parent_scores(mask), mask)
    lab = as_pairs(y, mask)
    hit = (par.argmax(axis=1) == lab.argmax(axis=1))[has_parent].mean()
    print(f"{label}: parent-heuristic pairwise acc {acc:.4f}  "
          f"(gen>=2 only: {hit:.4f}, on {has_parent.sum()} of {len(par)} trials)")
    return acc

parent_val  = parent_baseline(val_mask,  "Val ")
parent_test = parent_baseline(test_mask, "Test")

print("\nThe CNN sees geometry only, never is_parent_vase — but it must clear "
      f"{parent_test:.3f} to be worth more than one boolean.")

## Step 6 — Model Architecture: `VaseCNN`

The model reads one vase profile (two channels, `x` and `y`) and outputs a single logit — a raw score converted to a probability by a sigmoid.

```
Input:  (batch, 2, 125)        ← half-profile; 250 if HALF_CONTOUR = False
         ↓
Conv1d(2→32,   kernel=7) + BN + ReLU → MaxPool1d(2)   → (batch,  32, 62)
Conv1d(32→64,  kernel=5) + BN + ReLU → MaxPool1d(2)   → (batch,  64, 31)
Conv1d(64→128, kernel=3) + BN + ReLU → MaxPool1d(2)   → (batch, 128, 15)
Conv1d(128→256,kernel=3) + BN + ReLU                  → (batch, 256, 15)
AdaptiveAvgPool1d(1)                                  → (batch, 256, 1)
         ↓
Flatten → Linear(256→128) + ReLU + Dropout(0.1) → Linear(128→1)
Output: (batch,)  ← raw logit per vase
```

The cell prints the actual length after each stage, so the numbers above stay honest if you change the input.

### Padding mode follows the geometry
`PAD_MODE` comes from Step 3 and is not a tuning knob:

- **`"reflect"`** for the half-profile. It is an open curve whose ends sit on the symmetry axis, so the real continuation past each end *is* its reflection.
- **`"circular"`** for the full closed loop, where point 249 is genuinely adjacent to point 0.

Default zero padding would be wrong in both cases — it tells the network the curve begins from nothing and falls off a cliff, inventing discontinuities that are not in the shape.

### Capacity
Widened to 32→64→128→256 with dropout lowered to 0.1 after the first run showed train and validation metrics sitting exactly on top of each other for 19 epochs — zero generalisation gap, i.e. underfitting. Capacity was the binding constraint, not regularisation.

### Pooling
- **`MaxPool1d(2)`** halves the length, reducing computation and making features less position-sensitive.
- **`AdaptiveAvgPool1d(1)`** averages to one value per channel, giving a fixed `(batch, 256)` vector regardless of input length — which is what lets the same class serve both the 125- and 250-point inputs.

### How to tune

**More capacity** (train pairwise near validation and both low): a fifth conv block, wider channels, or a wider head.
**Less capacity** (train pulls away from validation): raise `Dropout`, add `weight_decay=1e-4` to Adam, or drop a conv block.
**Larger receptive field:** raise `kernel_size` in the early layers, or add `dilation=2`.


In [ ]:
class VaseCNN(nn.Module):
    """Conv trunk 32->64->128->256.

    Widened from 32->64->128 after the first run showed train and validation
    metrics sitting on top of each other for all 19 epochs — zero generalisation
    gap, i.e. the model was underfitting.

    `pad_mode` comes from Step 3 and is determined by the geometry: "reflect"
    for the open half-profile (its continuation past each end IS its reflection),
    "circular" for the full closed loop.
    """

    def __init__(self, in_channels=2, dropout=0.1, pad_mode=None):
        super().__init__()
        pad_mode = pad_mode or PAD_MODE
        self.pad_mode = pad_mode
        self.conv = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=7, padding=3, padding_mode=pad_mode),
            nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2, padding_mode=pad_mode),
            nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1, padding_mode=pad_mode),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1, padding_mode=pad_mode),
            nn.BatchNorm1d(256), nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),  # -> (batch, 256, 1); length-invariant
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.head(self.conv(x)).squeeze(1)


model = VaseCNN().to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}  "
      f"padding_mode='{model.pad_mode}'  input length {SEQ_LEN}")

# report the sequence length at each stage — the CAM resolution depends on it
with torch.no_grad():
    _h = torch.from_numpy(X[:2]).to(device)
    for i, layer in enumerate(model.conv):
        _h = layer(_h)
        if isinstance(layer, (nn.Conv1d, nn.MaxPool1d)):
            print(f"  conv[{i:2d}] {type(layer).__name__:<10} -> length {_h.shape[-1]}")

with torch.no_grad():
    _probe = model(torch.from_numpy(X[:4]).to(device))
assert _probe.shape == (4,), _probe.shape
print(f"Forward pass OK on {device} — output shape {tuple(_probe.shape)}")


## Step 7 — Training

### Loss function: `BCEWithLogitsLoss`
Binary Cross-Entropy with Logits. It applies a sigmoid internally, which is numerically more stable than sigmoid followed by a separate log loss. No `pos_weight` is needed: the classes are exactly balanced at 50/50.

### Optimiser: Adam
Adam adapts the learning rate per parameter from recent gradient history. Little tuning needed.

| Parameter | Default | Effect |
|---|---|---|
| `lr=1e-3` | Learning rate | Step size for each update. If loss is flat, try `1e-4`; if training crawls, try `3e-3`. |
| `weight_decay=0` | L2 penalty | Raise to `1e-4` if the train/val gap widens. |

### Learning-rate schedule: `ReduceLROnPlateau` — on **validation** loss
Watches validation loss; if it has not improved for `patience` epochs, multiplies the LR by `factor`. Feeding it *test* loss, as the original did, is model selection on the test set and quietly inflates the final number.

| Parameter | Default | Effect |
|---|---|---|
| `patience=3` | Epochs without improvement before reducing LR | Raise if validation loss is noisy |
| `factor=0.5` | LR multiplier when triggered | `0.5` halves it; `0.1` is more aggressive |

### Model selection and early stopping
We track the best **validation pairwise accuracy** seen so far and keep a deep copy of those weights. Training stops early if validation has not improved for `EARLY_STOP_PATIENCE` epochs. The final model is the best checkpoint, not whatever the last epoch happened to produce — so `EPOCHS` becomes an upper bound rather than a number you have to tune by eye.

### What to watch
Each epoch prints loss, thresholded accuracy, ROC-AUC and pairwise accuracy for train and validation.

- **Ignore thresholded accuracy near 0.500** — that is the chance floor, not a signal.
- **Watch `pair`** — validation pairwise accuracy is the task metric. Compare it to 0.500 (chance) and to the ~0.60 parent heuristic from Step 5.
- **Overfitting**: train pairwise climbs, validation pairwise stalls or falls → more dropout / weight decay / fewer epochs.
- **Underfitting**: both stay near 0.500 at the LR floor → more capacity, or the shape signal really is weak.

---

## Adding Demographic Data to Improve Performance

Vase preferences may differ systematically by age, gender, cultural background, and so on. Demographic features can be concatenated with the CNN's output vector before the final classifier.

### How to implement it

**1. Build a demographic feature array aligned with your trajectories:**
```python
# Shape: (n_trajectories, n_demo_features), row i describing participant_ids[i].
# Fit any scaler on TRAIN participants only, exactly as with NORM_STATS.
demo_features = ...
```

**2. Modify `VaseDataset` to carry demographic features:**
```python
class VaseDataset(Dataset):
    def __init__(self, X, y, demo):
        self.X    = torch.from_numpy(X)
        self.y    = torch.from_numpy(y)
        self.demo = torch.from_numpy(demo.astype(np.float32))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.demo[idx], self.y[idx]
```
The training and evaluation loops below unpack two items per batch, so they would need updating to unpack three.

**3. Modify `VaseCNN` to accept demographic features in the forward pass:**
```python
class VaseCNN(nn.Module):
    def __init__(self, in_channels=2, n_demo=5, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(...)   # unchanged
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 + n_demo, 64),  # ← 128 from CNN + n_demo features
            nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x, demo):
        cnn_out = self.conv(x).squeeze(-1)            # (batch, 128)
        combined = torch.cat([cnn_out, demo], dim=1)  # (batch, 128 + n_demo)
        return self.head(combined).squeeze(1)
```

### What demographic features are likely useful

| Feature | How to encode | Why it might help |
|---|---|---|
| Age | Single float, z-scored | Aesthetic preferences shift with age |
| Gender | One-hot (e.g. [1,0,0]) | Documented differences in shape preferences |
| Cultural background | One-hot or embedding | Cross-cultural aesthetics research shows systematic differences |
| Expertise / art training | Binary flag | Trained eyes may prefer different proportions |
| Decision time | Float (log-scaled) | Fast decisions may reflect gut preference; slow ones deliberation |

### Caveats
- **Demographics cannot improve pairwise accuracy on their own.** They are constant across both vases in a trial, so they shift both logits identically and leave the argmax unchanged. They only help through *interactions* with shape — which means the demographic vector has to reach the model somewhere it can modulate the shape features (e.g. FiLM-style conditioning on the conv output), not merely be concatenated before a single linear layer. Concatenation alone will move thresholded accuracy and AUC but not the metric you care about.
- Demographic values repeat for every trajectory belonging to a participant. Fine, but it means the effective sample size for those features is the number of *participants*, not trajectories — so they overfit easily.
- If demographics are missing for some participants, drop those participants or impute with the **training-set** mean.
- `decision_time_ms` is per-trial and available in the parquet, but it is measured *after* the choice is made. Using it to predict that same choice is leakage — exclude it unless you are deliberately modelling response time jointly.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.0)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=4, factor=0.5)


@torch.no_grad()
def predict(loader):
    """Logits for every row of an UNSHUFFLED loader, in loader order."""
    model.eval()
    return torch.cat([model(Xb.to(device)).float().cpu() for Xb, _ in loader]).numpy()


def metrics(loader, mask):
    """loss, thresholded accuracy, ROC-AUC, pairwise accuracy."""
    logits = predict(loader)
    labels = y[mask]
    loss = nn.functional.binary_cross_entropy_with_logits(
        torch.from_numpy(logits), torch.from_numpy(labels)
    ).item()
    acc = float(((logits > 0) == (labels > 0.5)).mean())
    auc = roc_auc_score(labels, logits)
    pair = pairwise_accuracy(logits, mask)
    return loss, acc, auc, pair


EPOCHS = 60
EARLY_STOP_PATIENCE = 10

# Select on validation AUC, not pairwise accuracy. Pairwise is a discrete argmax
# over ~10k trials (SE ~0.005) and bounced 0.622-0.649 epoch to epoch in the
# first run, which (a) fired early stopping while AUC was still climbing and
# (b) invited winner's curse — the chosen epoch was partly a lucky draw, which
# is most of that run's -0.009 val->test gap. AUC is threshold-free and smooth.
SELECT_ON = "val_auc"

best_val_auc = -np.inf
best_state, best_epoch = None, -1
epochs_since_best = 0
history = []

bar = tqdm(range(1, EPOCHS + 1), desc="training")
for epoch in bar:
    model.train()
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        criterion(model(X_batch), y_batch).backward()
        optimizer.step()

    tr_loss, tr_acc, tr_auc, tr_pair = metrics(train_eval_loader, train_mask)
    va_loss, va_acc, va_auc, va_pair = metrics(val_loader, val_mask)

    # LR schedule driven by VALIDATION loss — never test
    scheduler.step(va_loss)

    history.append({
        "epoch": epoch, "lr": optimizer.param_groups[0]["lr"],
        "train_loss": tr_loss, "train_acc": tr_acc, "train_auc": tr_auc, "train_pair": tr_pair,
        "val_loss": va_loss, "val_acc": va_acc, "val_auc": va_auc, "val_pair": va_pair,
    })

    flag = ""
    if va_auc > best_val_auc:
        best_val_auc, best_epoch = va_auc, epoch
        best_state = copy.deepcopy(model.state_dict())
        epochs_since_best = 0
        flag = "  *best*"
    else:
        epochs_since_best += 1

    tqdm.write(
        f"Epoch {epoch:2d}/{EPOCHS}  "
        f"train loss {tr_loss:.4f} acc {tr_acc:.3f} auc {tr_auc:.3f} pair {tr_pair:.3f}  |  "
        f"val loss {va_loss:.4f} acc {va_acc:.3f} auc {va_auc:.3f} pair {va_pair:.3f}{flag}"
    )
    bar.set_postfix(val_auc=f"{va_auc:.4f}", best=f"{best_val_auc:.4f}")

    if epochs_since_best >= EARLY_STOP_PATIENCE:
        tqdm.write(f"Early stopping — no validation AUC improvement for "
                   f"{EARLY_STOP_PATIENCE} epochs")
        break
bar.close()

model.load_state_dict(best_state)
history = pd.DataFrame(history)
best_row = history.loc[history.epoch == best_epoch].iloc[0]
print(f"\nRestored best checkpoint from epoch {best_epoch} "
      f"(val AUC {best_val_auc:.4f}, val pairwise {best_row.val_pair:.4f})")
best_val_pair = float(best_row.val_pair)

## Step 8 — Final Evaluation

The test set has not influenced anything up to this point: no normalisation statistics, no learning-rate decisions, no checkpoint selection. This cell touches it **once**.

### Metrics reported

| Metric | What it tells you |
|---|---|
| **Pairwise accuracy** | The task metric. Score both vases, pick the higher. Chance = 0.500. |
| **ROC-AUC** | Threshold-free ranking quality over all vases. Chance = 0.500. |
| **Thresholded accuracy** | Included for completeness, but near-useless here — the base rate is exactly 50%, so it sits near chance even for a working model. |
| **Brier score** | Mean squared error of the predicted probabilities. Measures *calibration* as well as ranking; chance = 0.250. |
| **Log loss** | The training objective. Chance = ln 2 = 0.6931. |

### Confidence intervals: cluster bootstrap by participant

Trials within a participant are **not independent** — the same person's 25 choices share their taste, and the parent vase carries over between generations. Resampling individual trials would therefore understate the uncertainty.

So the bootstrap resamples **participants** with replacement (409 in test), recomputing the metric over all trials of each drawn participant. That is the honest interval for "how would this do on a fresh sample of people".

The same resampling gives a paired interval on the **difference** CNN − parent heuristic, evaluated on identical trials within each bootstrap draw. The fraction of draws where the difference falls at or below zero is a one-sided p-value for "the CNN is no better than the heuristic".

A note on honesty: if you now go back, change the architecture, and re-run this cell, the test set has been used for model selection and stops being a clean estimate. Iterate on validation; re-run Step 8 when you are finished.

In [ ]:
CHANCE_LOGLOSS = float(np.log(2))

test_logits = predict(test_loader)
test_prob = 1 / (1 + np.exp(-test_logits))
test_labels = y[test_mask]

te_loss, te_acc, te_auc, te_pair = metrics(test_loader, test_mask)
te_brier = brier_score_loss(test_labels, test_prob)

cnn_correct = pairwise_correct(test_logits, test_mask)
par_correct = pairwise_correct(parent_scores(test_mask), test_mask)
test_gen = gen_of_trial["test"]
test_pid = pid_of_trial["test"]


def cluster_bootstrap(per_trial, pid, n_boot=5000, seed=SEED):
    """Resample PARTICIPANTS with replacement; return the bootstrap distribution
    of the mean of `per_trial`. Trials within a participant move together."""
    uniq, inv = np.unique(pid, return_inverse=True)
    sums = np.bincount(inv, weights=per_trial)
    counts = np.bincount(inv).astype(float)
    rng = np.random.default_rng(seed)
    draw = rng.integers(0, len(uniq), size=(n_boot, len(uniq)))
    return sums[draw].sum(axis=1) / counts[draw].sum(axis=1)


def ci(dist, lo=2.5, hi=97.5):
    return np.percentile(dist, lo), np.percentile(dist, hi)


boot_cnn  = cluster_bootstrap(cnn_correct, test_pid)
boot_par  = cluster_bootstrap(par_correct, test_pid)
boot_diff = cluster_bootstrap(cnn_correct - par_correct, test_pid)  # paired

cnn_lo, cnn_hi = ci(boot_cnn)
par_lo, par_hi = ci(boot_par)
dif_lo, dif_hi = ci(boot_diff)
p_one_sided = float((boot_diff <= 0).mean())

print("=" * 66)
print("TEST SET — 409 participants, 10,225 trials, 20,450 vases")
print("=" * 66)
print(f"{'metric':<26}{'value':>10}{'chance':>10}")
print("-" * 66)
print(f"{'pairwise accuracy':<26}{te_pair:>10.4f}{0.5:>10.4f}")
print(f"{'ROC-AUC':<26}{te_auc:>10.4f}{0.5:>10.4f}")
print(f"{'accuracy @ 0.5':<26}{te_acc:>10.4f}{0.5:>10.4f}")
print(f"{'Brier score':<26}{te_brier:>10.4f}{0.25:>10.4f}   (lower better)")
print(f"{'log loss':<26}{te_loss:>10.4f}{CHANCE_LOGLOSS:>10.4f}   (lower better)")
print()

print("Cluster bootstrap over participants (5,000 draws, 95% CI)")
print("-" * 66)
print(f"{'chance':<26}{0.5:>10.4f}")
print(f"{'parent heuristic':<26}{parent_test:>10.4f}   [{par_lo:.4f}, {par_hi:.4f}]")
print(f"{'VaseCNN (shape only)':<26}{te_pair:>10.4f}   [{cnn_lo:.4f}, {cnn_hi:.4f}]")
print("-" * 66)
print(f"{'CNN - chance':<26}{te_pair - 0.5:>+10.4f}")
print(f"{'CNN - parent (paired)':<26}{te_pair - parent_test:>+10.4f}   "
      f"[{dif_lo:+.4f}, {dif_hi:+.4f}]   p = {p_one_sided:.4f} (one-sided)")
print()

# --- Per-generation breakdown ---------------------------------------------
print("Pairwise accuracy by generation")
print("-" * 66)
print(f"{'gen':<6}{'trials':>8}{'CNN':>10}{'parent':>10}{'difference':>14}")
print("-" * 66)
gen_rows = []
for g in sorted(np.unique(test_gen)):
    sel = test_gen == g
    d = cluster_bootstrap(cnn_correct[sel] - par_correct[sel], test_pid[sel], n_boot=2000)
    row = {
        "generation": int(g), "n_trials": int(sel.sum()),
        "cnn": cnn_correct[sel].mean(), "parent": par_correct[sel].mean(),
        "cnn_lo": ci(cluster_bootstrap(cnn_correct[sel], test_pid[sel], n_boot=2000))[0],
        "cnn_hi": ci(cluster_bootstrap(cnn_correct[sel], test_pid[sel], n_boot=2000))[1],
        "diff_lo": ci(d)[0], "diff_hi": ci(d)[1],
    }
    gen_rows.append(row)
    note = "  (no parent exists)" if g == 1 else ""
    print(f"{row['generation']:<6}{row['n_trials']:>8}{row['cnn']:>10.4f}"
          f"{row['parent']:>10.4f}{row['cnn'] - row['parent']:>+14.4f}{note}")
gen_table = pd.DataFrame(gen_rows)

# --- Confusion matrix at the 0.5 threshold --------------------------------
cm = confusion_matrix(test_labels.astype(int), (test_prob > 0.5).astype(int))
print()
print("Confusion matrix @ 0.5 (rows = actual, cols = predicted)")
print("-" * 66)
print(f"{'':<14}{'pred not sel':>14}{'pred selected':>15}")
print(f"{'not selected':<14}{cm[0,0]:>14,}{cm[0,1]:>15,}")
print(f"{'selected':<14}{cm[1,0]:>14,}{cm[1,1]:>15,}")

print(f"\nValidation: AUC {best_val_auc:.4f}, pairwise {best_val_pair:.4f} "
      f"(test - val pairwise: {te_pair - best_val_pair:+.4f})")

## Step 9 — Evaluation Plots

Four figures, written to `figures/vase_select_cnn/`. They answer four different questions, in order of how much you should trust them.

| Figure | Question |
|---|---|
| `01_training_curves` | Did training converge, and did it overfit? |
| `02_test_diagnostics` | Does the model rank vases well, and are its probabilities honest? |
| `03_model_comparison` | Is it better than the baselines, and where does its edge come from? |
| `04_what_the_model_likes` | **What shape has it actually learned to prefer?** |

Figure 4 is the one with scientific content rather than engineering content. A pairwise accuracy is a number about a model; the contours it scores highest and lowest are a claim about vase aesthetics that you can look at and argue with.

### On averaging contours
The bottom panel of figure 4 averages contours pointwise across many vases. That is only legitimate because every contour is arc-length parameterised, has the same 250 points, starts at the same canonical location (x ≈ 0 on the rim) and winds the same direction — all verified in Step 2. Point *i* means the same place on the silhouette in every vase, so the pointwise mean is a real average shape rather than mush.

### Chart conventions
Sequential blue for the model, orange for the comparison series, gray for de-emphasised baselines; hairline gridlines; chance drawn as a muted reference rule rather than a series. Same palette as `density_diff/pc_density_diff_map.ipynb`.

In [ ]:
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "savefig.facecolor": SURFACE, "savefig.bbox": "tight", "savefig.dpi": 200,
})


def style(ax, title=None, xlabel=None, ylabel=None, grid="y"):
    """Recessive chrome: hairline grid, no top/right spines, muted ticks."""
    if title:
        ax.set_title(title, color=INK, pad=8)
    if xlabel:
        ax.set_xlabel(xlabel, color=MUTED)
    if ylabel:
        ax.set_ylabel(ylabel, color=MUTED)
    ax.tick_params(labelsize=8, colors=MUTED, length=2)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    if grid:
        ax.grid(axis=grid, color=GRID, lw=0.6, alpha=0.9)
    ax.set_axisbelow(True)


def chance_rule(ax, level=0.5, label="chance", axis="h"):
    """Chance is a reference, not a series — muted hairline, labelled once.
    The label sits on the LEFT so it cannot collide with right-edge annotations."""
    line = ax.axhline if axis == "h" else ax.axvline
    line(level, color=MUTED, lw=0.9, ls="-", alpha=0.7, zorder=1)
    if label:
        ax.annotate(label, xy=(0.006, level), xycoords=("axes fraction", "data"),
                    ha="left", va="bottom", fontsize=7.5, color=MUTED)


print("Plot helpers ready. Figures →", FIG_DIR.resolve())

In [ ]:
"""Figure 1 — training curves. Three panels because the three quantities live on
different scales; never two y-axes on one plot."""

fig, axes = plt.subplots(1, 3, figsize=(13.5, 3.9))
panels = [
    ("train_loss", "val_loss",  "Log loss",          CHANCE_LOGLOSS, "chance = ln 2"),
    ("train_auc",  "val_auc",   "ROC-AUC",           0.5,            "chance"),
    ("train_pair", "val_pair",  "Pairwise accuracy", 0.5,            "chance"),
]

for ax, (tr_col, va_col, title, ref, ref_label) in zip(axes, panels):
    ax.plot(history.epoch, history[tr_col], color=BLUE, lw=2, label="train", zorder=3)
    ax.plot(history.epoch, history[va_col], color=ORANGE, lw=2, label="validation", zorder=3)
    chance_rule(ax, ref, ref_label)

    # The selected checkpoint is a rule plus a caret on the axis — no floating
    # text label, which would collide with the curves or the chance label
    # depending on where the best epoch happens to land.
    ax.axvline(best_epoch, color=MUTED, lw=0.9, alpha=0.7, zorder=1)
    ax.plot([best_epoch], [0], marker="^", ms=6, color=MUTED,
            transform=ax.get_xaxis_transform(), clip_on=False, zorder=5)

    style(ax, title, "epoch", None)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.legend(frameon=False, fontsize=8, labelcolor=MUTED, loc="best")

# direct-label the endpoint of the series that matters
va_last = history.val_pair.iloc[-1]
axes[2].annotate(f"{va_last:.3f}", xy=(history.epoch.iloc[-1], va_last),
                 xytext=(5, 0), textcoords="offset points",
                 fontsize=8, color=ORANGE, va="center")

fig.suptitle(
    f"Training history — VaseCNN   (▲ selected epoch {best_epoch} "
    f"of {int(history.epoch.max())}, chosen on validation AUC)",
    fontsize=11, color=INK, y=1.03)
fig.tight_layout()
fig.savefig(FIG_DIR / "01_training_curves.png")
plt.show()

gap = history.train_pair.iloc[-1] - history.val_pair.iloc[-1]
print(f"Final train-val pairwise gap: {gap:+.4f}  "
      f"({'overfitting' if gap > 0.02 else 'no meaningful overfitting'})")

In [ ]:
"""Figure 2 — test-set diagnostics: ranking quality, calibration, separation."""

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.1))

# --- ROC ------------------------------------------------------------------
ax = axes[0]
fpr, tpr, _ = roc_curve(test_labels, test_logits)
ax.plot([0, 1], [0, 1], color=MUTED, lw=0.9, alpha=0.7, zorder=1)
ax.annotate("chance", xy=(0.62, 0.58), fontsize=7.5, color=MUTED, rotation=45)
ax.plot(fpr, tpr, color=BLUE, lw=2, zorder=3)
ax.fill_between(fpr, fpr, tpr, color=BLUE, alpha=0.10, zorder=2)
ax.annotate(f"AUC = {te_auc:.3f}", xy=(0.55, 0.22), fontsize=10, color=INK)
style(ax, "ROC curve", "false positive rate", "true positive rate", grid="both")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")

# --- Calibration ----------------------------------------------------------
# Quantile bins: the predicted probabilities occupy a narrow band, so equal-width
# bins would leave the tails nearly empty.
ax = axes[1]
n_bins = 10
edges = np.quantile(test_prob, np.linspace(0, 1, n_bins + 1))
edges[-1] += 1e-9
idx = np.digitize(test_prob, edges[1:-1])
mean_pred = np.array([test_prob[idx == b].mean() for b in range(n_bins)])
obs_freq = np.array([test_labels[idx == b].mean() for b in range(n_bins)])
bin_n = np.array([(idx == b).sum() for b in range(n_bins)])
se = np.sqrt(obs_freq * (1 - obs_freq) / bin_n)

lim = (min(mean_pred.min(), obs_freq.min()) - 0.02,
       max(mean_pred.max(), obs_freq.max()) + 0.02)
ax.plot(lim, lim, color=MUTED, lw=0.9, alpha=0.7, zorder=1)
ax.annotate("perfect calibration", xy=(0.04, 0.9), xycoords="axes fraction",
            fontsize=7.5, color=MUTED)
ax.errorbar(mean_pred, obs_freq, yerr=1.96 * se, fmt="o", color=BLUE,
            ms=6, lw=1.6, capsize=0, mec=SURFACE, mew=1.5, zorder=3)
ax.plot(mean_pred, obs_freq, color=BLUE, lw=1.4, alpha=0.6, zorder=2)
style(ax, "Calibration (10 quantile bins)",
      "mean predicted probability", "observed selection rate", grid="both")
ax.set_xlim(*lim); ax.set_ylim(*lim); ax.set_aspect("equal")
ax.annotate(f"Brier = {te_brier:.4f}", xy=(0.97, 0.06), xycoords="axes fraction",
            ha="right", fontsize=9, color=INK)

# --- Score separation -----------------------------------------------------
ax = axes[2]
bins = np.linspace(test_prob.min(), test_prob.max(), 60)
for lab, colour, name in [(0, ORANGE, "not selected"), (1, BLUE, "selected")]:
    ax.hist(test_prob[test_labels == lab], bins=bins, histtype="step",
            lw=1.8, color=colour, label=name, density=True, zorder=3)
chance_rule(ax, 0.5, None, axis="v")
# label the 0.5 rule at the BOTTOM so it clears the separation annotation
ax.annotate("0.5", xy=(0.5, 0.02), xycoords=("data", "axes fraction"),
            xytext=(3, 0), textcoords="offset points",
            fontsize=7.5, color=MUTED, ha="left", va="bottom")
style(ax, "Predicted probability by outcome", "predicted probability", "density")
ax.legend(frameon=False, fontsize=8, labelcolor=MUTED, loc="upper left")
sep = test_prob[test_labels == 1].mean() - test_prob[test_labels == 0].mean()
ax.annotate(f"mean separation {sep:+.4f}", xy=(0.97, 0.94), xycoords="axes fraction",
            ha="right", fontsize=8.5, color=INK)

fig.suptitle("Test-set diagnostics — 20,450 vases from 409 unseen participants",
             fontsize=11, color=INK, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "02_test_diagnostics.png")
plt.show()

print(f"Calibration: max |predicted - observed| across bins = "
      f"{np.abs(mean_pred - obs_freq).max():.4f}")

In [ ]:
"""Figure 3 — is it better than the baselines, and where does the edge come from?"""

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3),
                         gridspec_kw={"width_ratios": [1, 1.25]})

# --- Model comparison: emphasis, not categorical --------------------------
# One measure across three models -> one hue, with the model that is the point
# in the accent colour and the baselines receding to gray.
ax = axes[0]
names = ["chance", "parent\nheuristic", "VaseCNN\n(shape only)"]
vals = [0.5, parent_test, te_pair]
errs = [(0, 0),
        (parent_test - par_lo, par_hi - parent_test),
        (te_pair - cnn_lo, cnn_hi - te_pair)]
colours = [GRAY_MARK, GRAY_MARK, BLUE]

ax.bar(names, vals, color=colours, width=0.55, zorder=3)
ax.errorbar(range(3), vals, yerr=np.array(errs).T, fmt="none",
            ecolor=INK, elinewidth=1.2, capsize=0, alpha=0.55, zorder=4)
for i, v in enumerate(vals):
    ax.annotate(f"{v:.3f}", xy=(i, v), xytext=(0, 9), textcoords="offset points",
                ha="center", fontsize=9.5, color=INK)

style(ax, "Test pairwise accuracy", None, "pairwise accuracy")
ax.set_ylim(0.45, max(vals) + 0.07)
ax.axhline(0.5, color=MUTED, lw=0.9, alpha=0.7, zorder=1)
ax.annotate(f"CNN − parent = {te_pair - parent_test:+.4f}\n"
            f"95% CI [{dif_lo:+.4f}, {dif_hi:+.4f}]",
            xy=(0.03, 0.96), xycoords="axes fraction", va="top",
            fontsize=8.5, color=MUTED)

# --- Per-generation -------------------------------------------------------
ax = axes[1]
g = gen_table.generation
ax.fill_between(g, gen_table.cnn_lo, gen_table.cnn_hi,
                color=BLUE, alpha=0.13, lw=0, zorder=2)
ax.plot(g, gen_table.cnn, color=BLUE, lw=2, marker="o", ms=7,
        mec=SURFACE, mew=1.5, label="VaseCNN", zorder=4)
ax.plot(g, gen_table.parent, color=ORANGE, lw=2, marker="o", ms=7,
        mec=SURFACE, mew=1.5, label="parent heuristic", zorder=3)

# chance label goes on the RIGHT here — the left edge is occupied by the
# generation-1 marker, which sits exactly on 0.5
chance_rule(ax, 0.5, None)
ax.annotate("chance", xy=(0.994, 0.5), xycoords=("axes fraction", "data"),
            ha="right", va="bottom", fontsize=7.5, color=MUTED)

# gen 1 has no parent — say so rather than letting the reader infer a collapse
ax.annotate("no parent exists\nat generation 1", xy=(1, gen_table.parent.iloc[0]),
            xytext=(12, 10), textcoords="offset points",
            fontsize=7.5, color=MUTED, va="bottom")

style(ax, "Pairwise accuracy by generation", "generation", "pairwise accuracy")
ax.set_xticks(g)
ax.legend(frameon=False, fontsize=8.5, labelcolor=MUTED, loc="upper right")

fig.suptitle("VaseCNN vs baselines — test set, cluster-bootstrap CIs over participants",
             fontsize=11, color=INK, y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "03_model_comparison.png")
plt.show()

print(gen_table.round(4).to_string(index=False))

In [ ]:
"""Figure 4 — what shape has the model actually learned to prefer?

Ranked over DISTINCT shapes. The test split contains the same contour more than
once: a vase selected at generation N reappears as the parent at N+1, so it is
scored identically each time. Ranking raw rows put five copies of one
participant's vase in the top six. Deduplicating on `shape_hash` first gives six
genuinely different shapes, and weights each shape once in the decile means.

Top rows: the individual test contours it scores highest and lowest.
Bottom: the pointwise mean contour of the top and bottom deciles, which is only
meaningful because all contours are arc-length parameterised from a common
start point with consistent winding (verified in Step 2)."""

assert len(shape_hash) == len(y), (
    "shape_hash is out of sync with the data — re-run Step 3 "
    "(and leave RESTRICT/DROP flags off, or filter shape_hash alongside)"
)

# --- Rank over distinct shapes -------------------------------------------
DEDUPE_RANKING = True   # False reproduces the old row-level ranking

test_hash = shape_hash[test_mask]
if DEDUPE_RANKING:
    _, first = np.unique(test_hash, return_index=True)
    keep_idx = np.sort(first)           # first occurrence, original order
else:
    keep_idx = np.arange(len(test_logits))

contours = X_raw[test_mask][keep_idx]   # (n, 2, 250) in ORIGINAL units
rank_logits = test_logits[keep_idx]
rank_prob = test_prob[keep_idx]
order = np.argsort(rank_logits)
N_SHOW = 6

print(f"Ranking over {len(keep_idx):,} distinct shapes "
      f"(from {len(test_logits):,} test rows)")

def draw(ax, xy, colour, lw=1.6, alpha=1.0, zorder=3):
    """Close the loop explicitly so the seam doesn't show as a gap."""
    x = np.append(xy[0], xy[0][0])
    yv = np.append(xy[1], xy[1][0])
    ax.plot(x, yv, color=colour, lw=lw, alpha=alpha, zorder=zorder,
            solid_joinstyle="round")

fig = plt.figure(figsize=(13.5, 8.6))
gs = fig.add_gridspec(3, N_SHOW, height_ratios=[1, 1, 1.5],
                      hspace=0.35, wspace=0.6)

# common limits so the small multiples are comparable to each other
extremes = np.concatenate([order[-N_SHOW:], order[:N_SHOW]])
xl = (contours[extremes, 0].min() - 0.3, contours[extremes, 0].max() + 0.3)
yl = (contours[extremes, 1].min() - 0.3, contours[extremes, 1].max() + 0.3)

shown = []
for col in range(N_SHOW):
    # row 0: highest-scored
    i = order[-(col + 1)]
    ax = fig.add_subplot(gs[0, col])
    draw(ax, contours[i], BLUE)
    ax.set_title(f"p = {rank_prob[i]:.3f}", fontsize=8, color=MUTED, pad=4)
    ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_aspect("equal"); ax.axis("off")
    if col == 0:
        ax.text(-0.12, 0.5, "most preferred", transform=ax.transAxes, rotation=90,
                va="center", ha="right", fontsize=9, color=BLUE)
    shown.append(i)

    # row 1: lowest-scored
    j = order[col]
    ax = fig.add_subplot(gs[1, col])
    draw(ax, contours[j], ORANGE)
    ax.set_title(f"p = {rank_prob[j]:.3f}", fontsize=8, color=MUTED, pad=4)
    ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_aspect("equal"); ax.axis("off")
    if col == 0:
        ax.text(-0.12, 0.5, "least preferred", transform=ax.transAxes, rotation=90,
                va="center", ha="right", fontsize=9, color=ORANGE)
    shown.append(j)

# the whole point of the dedup: every panel must be a different shape
assert len(set(test_hash[keep_idx][shown])) == len(shown), "a shape is shown twice"

# --- Mean contour of top vs bottom decile ---------------------------------
k = len(order) // 10
top_mean = contours[order[-k:]].mean(axis=0)
bot_mean = contours[order[:k]].mean(axis=0)
all_mean = contours.mean(axis=0)

ax = fig.add_subplot(gs[2, :3])
draw(ax, all_mean, GRAY_MARK, lw=1.4, zorder=2)
draw(ax, bot_mean, ORANGE, lw=2.4)
draw(ax, top_mean, BLUE, lw=2.4)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title(f"Mean contour — top vs bottom decile ({k:,} distinct shapes each)",
             fontsize=10, color=INK, pad=10)
# legend BELOW the panel: to the right it collides with the next panel's axis
ax.legend(handles=[
    Line2D([], [], color=BLUE, lw=2.4, label="top decile (most preferred)"),
    Line2D([], [], color=ORANGE, lw=2.4, label="bottom decile"),
    Line2D([], [], color=GRAY_MARK, lw=1.4, label="all distinct test shapes"),
], frameon=False, fontsize=8.5, labelcolor=MUTED, ncol=3,
   loc="upper center", bbox_to_anchor=(0.5, -0.02))

# --- Width profile: where along the height do they differ? ----------------
# Half-width at each height band, which is what "shape preference" reduces to
# for a silhouette.
ax = fig.add_subplot(gs[2, 3:])
def half_width(mean_xy, n_bands=60):
    xs, ys = mean_xy[0], mean_xy[1]
    edges = np.linspace(ys.min(), ys.max(), n_bands + 1)
    b = np.clip(np.digitize(ys, edges[1:-1]), 0, n_bands - 1)
    centres = 0.5 * (edges[:-1] + edges[1:])
    w = np.array([np.abs(xs[b == i]).max() if (b == i).any() else np.nan
                  for i in range(n_bands)])
    return centres, w

for mean_xy, colour, name in [(top_mean, BLUE, "top decile"),
                              (bot_mean, ORANGE, "bottom decile")]:
    c, w = half_width(mean_xy)
    ax.plot(w, c, color=colour, lw=2, zorder=3, label=name)
style(ax, "Half-width profile of the mean contour",
      "half-width (radius)", "height", grid="both")
ax.legend(frameon=False, fontsize=8.5, labelcolor=MUTED, loc="best")

fig.suptitle("What the model prefers — distinct test shapes ranked by predicted probability",
             fontsize=11, color=INK, y=0.98)
fig.savefig(FIG_DIR / "04_what_the_model_likes.png")
plt.show()

for name, m in [("Top decile   ", top_mean), ("Bottom decile", bot_mean)]:
    h = float(np.ptp(m[1]))
    w = float(np.abs(m[0]).max())
    print(f"{name} — height {h:.3f}, max half-width {w:.3f}, "
          f"height/width ratio {h / (2 * w):.3f}")
print("\n(ratio = height / width of the average contour — the headline shape difference)")


In [ ]:
"""Step 10 — save everything the comparison model will need.

The split participant IDs and NORM_STATS are the important part: a pairwise /
siamese model trained later must be evaluated on the SAME test participants
with the SAME normalisation, or the comparison is meaningless."""

CHECKPOINT = "vase_select_cnn.pt"
torch.save(
    {
        "state_dict": best_state,
        "arch": "VaseCNN 32-64-128-256, circular padding, dropout 0.1",
        "best_epoch": best_epoch,
        "select_on": SELECT_ON,
        "norm_stats": NORM_STATS,
        "seed": SEED,
        "train_pids": train_pids,
        "val_pids": val_pids,
        "test_pids": test_pids,
        "val_metrics": {"auc": best_val_auc, "pairwise": best_val_pair},
        "test_metrics": {
            "loss": te_loss, "acc": te_acc, "auc": te_auc,
            "pairwise": te_pair, "brier": te_brier,
            "pairwise_ci": (cnn_lo, cnn_hi),
        },
        "parent_baseline_test": parent_test,
        "parent_baseline_ci": (par_lo, par_hi),
        "diff_vs_parent": {"estimate": te_pair - parent_test,
                           "ci": (dif_lo, dif_hi), "p_one_sided": p_one_sided},
        "test_logits": test_logits,      # so a later model can be compared per-trial
        "gen_table": gen_table,
        "history": history,
    },
    CHECKPOINT,
)
print(f"Saved {CHECKPOINT}")
print(f"Figures written to {FIG_DIR.resolve()}:")
for p in sorted(FIG_DIR.glob('*.png')):
    print(f"  {p.name}")

## Step 11 — Explainable AI: which parts of the silhouette drive the decision

Two attribution methods, because they answer different questions and cross-check each other.

| Method | Resolution | What it is faithful to |
|---|---|---|
| **CAM at conv4** | 15 positions over the 125-point profile | The **decision**. This is the layer the classifier head actually reads. |
| **CAM at conv2** | 62 positions | **Space.** Sharper localisation, but of lower-level features that may not survive to the output. |
| **Integrated Gradients** | True per-point, all 125 coordinates | The **input**. Axiomatic (completeness), independent of layer choice. |

conv4 is not "the most accurate" — it is the most decision-faithful and the *least* spatially precise. Reading both is the point. The compute cell prints how many input points each position covers.

### Three implementation choices worth knowing

**1. It is CAM, not Grad-CAM.** Because the trunk ends in `AdaptiveAvgPool1d(1)` before the head, the gradient of the logit with respect to a feature map is *constant across positions* within a channel. Grad-CAM's channel weights therefore reduce exactly to the original CAM weights. Nothing is lost, but the notebook calls it what it is.

**2. The maps are signed — no ReLU.** Standard Grad-CAM keeps only evidence *for* the class. Here we want both directions: what made a vase attractive *and* what made it repellent. Signed maps on a diverging blue↔red ramp with a neutral midpoint, so "this region did not matter" reads as nothing rather than as a third category.

**3. Attribution is projected onto the outward normal.** A raw input gradient is a 2-vector per point and hard to read. Projected onto the local outward normal it becomes a signed scalar with a physical meaning:

> *If this point bulged outward, would the vase become more or less likely to be selected?*

That turns the explanation from "this region matters" into a claim about vase aesthetics. The projection is valid in either coordinate space, because Step 4 normalises with a per-channel shift and a **single shared scale** — a similarity transform, which preserves angles. Normals are computed on the full closed contour, where the central difference is well defined everywhere, then sliced to the half.

The IG baseline is the **mean training profile** — a real average vase — so attributions read as "which deviations from a typical vase moved the score", not as deviations from an all-zeros input that is not a vase at all. The completeness check printed below should be small relative to the logit range; if it is not, raise `IG_STEPS`.

### Why this section is simpler than it used to be

An earlier version of this notebook fed the model the full 250-point closed contour, and the attribution maps came out almost entirely one-sided — the input gradient was ~7× larger on one half than the other, in every vase.

That was not a property of the model worth reporting. It was the expected consequence of feeding a **duplicated signal**: the outline traces a mirror-symmetric profile twice, so the two halves are the same curve, and the network simply used one copy and ignored the other. Which copy it picked was arbitrary.

Since Step 3 now feeds only the half-profile, the redundancy is gone and every position in these maps carries independent signal. Attributions are mirrored back onto the whole outline for display, which is exact.

### How to read the figures

- **Figure 5** paints attribution onto individual vases, one row selected and one rejected, sampled at fixed score quantiles rather than at the extremes so the panel is representative. Test AUC is ~0.60, so **any single vase's map is noisy** — do not over-read one panel.
- **Figure 6** is the stable result: thousands of vases averaged, with bootstrap bands resampled over *participants* (same reasoning as Step 8 — a person's 25 choices are not independent). Panel A folds attribution onto height, which is the physically meaningful question; panel B shows the same as a pressure field; panel C is the raw profile from rim to foot.

A caveat on panel A: the selected and rejected curves are expected to look similar. This measures what the **model** responds to, not a contrast between the two groups.


In [ ]:
"""Step 11 — Attribution: which parts of the silhouette drive the decision.

Two attribution methods, because they answer different questions:

  CAM (conv4)  — faithful to the DECISION: the layer the classifier head reads.
                 Coarse in space (large receptive field).
  CAM (conv2)  — sharper in space, but further from the output, so it reflects
                 lower-level features that may not survive to the decision.
  Integrated   — true per-point attribution on the raw (x, y) input, relative to
  Gradients      a baseline vase. Axiomatic, and independent of layer choice.

Naming: because the trunk ends in AdaptiveAvgPool1d before the head, the gradient
of the logit w.r.t. a feature map is CONSTANT across positions within a channel.
Grad-CAM therefore collapses exactly to the original CAM. This notebook says CAM.

Maps are SIGNED (no ReLU): we want what pushed a vase toward rejection too.

With HALF_CONTOUR the model sees only the right half, so attributions are length
SEQ_LEN (125). `to_full()` mirrors them back onto all 250 points for plotting,
which is exact — the two halves are the same curve.
"""

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import LineCollection

# Same diverging ramp as density_diff/pc_density_diff_map.ipynb: two opposite
# hues with a NEUTRAL midpoint, so "this region did not matter" reads as nothing.
DIVERGING = LinearSegmentedColormap.from_list("blue_red", [
    "#0d366b", "#184f95", "#2a78d6", "#86b6ef", "#cde2fb",
    "#f0efec",
    "#f7c9c8", "#ee9695", "#e34948", "#bd2f2e", "#8f1f1f"])

assert "model" in dir(), "run Step 7 first — `model` must hold the trained weights"
model.eval()

RELU_IDX = [i for i, l in enumerate(model.conv) if isinstance(l, nn.ReLU)]
CONV4_IDX, CONV2_IDX = RELU_IDX[3], RELU_IDX[1]
print(f"CAM hooks — conv4 at model.conv[{CONV4_IDX}], conv2 at model.conv[{CONV2_IDX}]")

# full-contour index -> index into a half-profile array (0..SEQ_LEN-1)
_ar = np.arange(FULL_LEN)
HALF_TO_FULL = (np.where((_ar >= 1) & (_ar <= 125), _ar, MIRROR) - 1)


def to_full(attr):
    """Mirror a half-profile attribution onto all 250 contour points."""
    return attr[..., HALF_TO_FULL] if HALF_CONTOUR else attr


# --------------------------------------------------------------------------
# 1. CAM
# --------------------------------------------------------------------------
def cam_maps(x, layer_idx, batch_size=256):
    """Signed CAM at `model.conv[layer_idx]`. Returns (n, L) where L is that
    layer's sequence length, NOT SEQ_LEN."""
    layer = model.conv[layer_idx]
    out_maps, store = [], {}
    fh = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    bh = layer.register_full_backward_hook(
        lambda m, gi, go: store.__setitem__("g", go[0]))
    try:
        for s in range(0, len(x), batch_size):
            xb = torch.from_numpy(x[s:s + batch_size]).to(device)
            model.zero_grad(set_to_none=True)
            # samples are independent, so one backward serves the whole batch
            model(xb).sum().backward()
            A, G = store["a"].detach(), store["g"].detach()
            alpha = G.mean(dim=2, keepdim=True)              # (B, C, 1)
            out_maps.append((alpha * A).sum(dim=1).float().cpu().numpy())
    finally:
        fh.remove(); bh.remove()
        model.zero_grad(set_to_none=True)
    return np.concatenate(out_maps)


def upsample_to_input(maps, out_len=None, circular=None):
    """Interpolate a length-L map onto the model's input positions.

    Wraps only when the input is the closed loop; the half-profile is an open
    curve, so wrapping would join the rim to the foot. This is INTERPOLATION —
    it does not recover resolution the layer never had."""
    out_len = out_len or SEQ_LEN
    circular = (not HALF_CONTOUR) if circular is None else circular
    n, L = maps.shape
    src = (np.arange(L) + 0.5) * (out_len / L)
    dst = np.arange(out_len)
    kw = {"period": out_len} if circular else {}
    return np.stack([np.interp(dst, src, m, **kw) for m in maps])


# --------------------------------------------------------------------------
# 2. Integrated Gradients
# --------------------------------------------------------------------------
def integrated_gradients(x, baseline, steps=64, batch_size=256):
    """IG w.r.t. the input, baseline = an average vase. Returns (n, 2, SEQ_LEN)."""
    total = np.zeros_like(x)
    base_t = torch.from_numpy(baseline).to(device)
    for s in range(0, len(x), batch_size):
        xb = torch.from_numpy(x[s:s + batch_size]).to(device)
        acc = torch.zeros_like(xb)
        for a in np.linspace(1.0 / steps, 1.0, steps):     # right Riemann sum
            xi = (base_t + float(a) * (xb - base_t)).detach().requires_grad_(True)
            g, = torch.autograd.grad(model(xi).sum(), xi)
            acc += g
        total[s:s + batch_size] = ((xb - base_t) * acc / steps).float().cpu().numpy()
    return total


def outward_normals(xy_full):
    """Unit outward normal at each point of the FULL closed contour (2, 250).

    Orientation comes from the polygon's SIGNED AREA, not from a centroid test.
    A centroid test asks whether the normal points away from the centre, which
    is wrong wherever the outline is non-convex: at a waisted neck the true
    outward normal can point back toward the centroid. Measured on this data,
    the centroid test flipped 5.8% of points on average, affected 58% of vases,
    and reached 48% of points on the worst one — concentrated at exactly the
    necks and shoulders where the signal lives. The winding rule is exact for
    any simple closed polygon.

    Computed on the closed loop so the central difference is well defined
    everywhere; slice afterwards for the half. Raw and normalised coordinates
    give identical directions, because Step 4 normalises with a per-channel
    shift and a SINGLE shared scale — a similarity transform, which preserves
    angles.
    """
    P = xy_full.T
    T = np.roll(P, -1, axis=0) - np.roll(P, 1, axis=0)
    T /= np.linalg.norm(T, axis=1, keepdims=True) + 1e-9
    N = np.stack([T[:, 1], -T[:, 0]], axis=1)   # outward for counter-clockwise
    area = 0.5 * np.sum(P[:, 0] * np.roll(P[:, 1], -1)
                        - np.roll(P[:, 0], -1) * P[:, 1])
    return N * np.sign(area)


def input_gradients(x, batch_size=256):
    """Plain d(logit)/d(input) evaluated AT x. Returns (n, 2, SEQ_LEN)."""
    out = np.empty_like(x)
    for s in range(0, len(x), batch_size):
        xb = torch.from_numpy(x[s:s + batch_size]).to(device).requires_grad_(True)
        g, = torch.autograd.grad(model(xb).sum(), xb)
        out[s:s + batch_size] = g.float().cpu().numpy()
    return out


def project_normal(field, contours_full):
    """Project a per-point 2-vector field onto the outward normal."""
    out = np.empty((len(field), field.shape[2]), dtype=np.float32)
    for i in range(len(field)):
        N = outward_normals(contours_full[i])
        out[i] = np.einsum("ci,ic->i", field[i], N[HALF] if HALF_CONTOUR else N)
    return out


# Two DIFFERENT quantities. Conflating them is an easy and serious mistake:
#
#   SENSITIVITY  = project_normal(input_gradients(x))
#       d(logit) / d(outward displacement), evaluated at this vase.
#       Positive => bulging out HERE would raise the score.
#       This is the one that answers "which way should the shape move?".
#
#   CONTRIBUTION = project_normal(ig)
#       IG is (x - baseline) * mean-gradient-along-the-path, so its sign mixes
#       WHICH WAY this point deviates from the average vase with WHICH WAY
#       helps. Positive => this point's deviation from an average vase pushed
#       the score up. It does NOT mean "bulging out here would help".
#
# Use sensitivity for direction, |contribution| or |sensitivity| for importance.


In [ ]:
"""Run the attribution methods over the test split."""

X_test    = X[test_mask]              # model input (half-profile, normalised)
full_test = X_full[test_mask]         # full closed contour, raw units, for plots
sel_test  = y[test_mask]
pid_test  = participant_ids[test_mask]
hash_test = shape_hash[test_mask]

# --- CAM at both depths ----------------------------------------------------
cam4_raw = cam_maps(X_test, CONV4_IDX)
cam2_raw = cam_maps(X_test, CONV2_IDX)
for nm, m in [("conv4", cam4_raw), ("conv2", cam2_raw)]:
    print(f"CAM {nm}: {m.shape}  -> {SEQ_LEN / m.shape[1]:.1f} input points per position")

cam4 = upsample_to_input(cam4_raw)    # (n, SEQ_LEN)
cam2 = upsample_to_input(cam2_raw)

# --- Integrated Gradients on a subsample -----------------------------------
# Baseline is the mean TRAINING profile — a real average vase — so attributions
# read as "which deviations from a typical vase moved the score", rather than
# being measured against an all-zeros input that is not a vase at all.
N_IG = 4000
IG_STEPS = 64
baseline = X[train_mask].mean(axis=0, keepdims=True).astype(np.float32)

rng = np.random.default_rng(SEED)
ig_idx = np.sort(rng.choice(len(X_test), size=min(N_IG, len(X_test)), replace=False))
ig = integrated_gradients(X_test[ig_idx], baseline, steps=IG_STEPS)

# Completeness axiom: IG should sum to f(x) - f(baseline). A large violation
# means IG_STEPS is too low and the attributions are not trustworthy.
with torch.no_grad():
    f_x = model(torch.from_numpy(X_test[ig_idx]).to(device)).float().cpu().numpy()
    f_b = float(model(torch.from_numpy(baseline).to(device)).float().cpu().numpy()[0])
err = np.abs(ig.sum(axis=(1, 2)) - (f_x - f_b))
print(f"\nIG completeness — median |error| {np.median(err):.4f}, max {err.max():.4f} "
      f"(logit range being explained: {np.ptp(f_x):.3f})")

# --- Project onto the outward normal --------------------------------------
# SENSITIVITY: d(logit)/d(outward displacement) at this vase. This is the map
# that answers "should this part be wider or narrower?".
nsens = project_normal(input_gradients(X_test[ig_idx]), full_test[ig_idx])
# CONTRIBUTION: IG projected the same way. Importance, not direction — see the
# note in the previous cell before reading a sign off this one.
nattr = project_normal(ig, full_test[ig_idx])
print(f"Normal sensitivity : {nsens.shape}  (+ = bulging out here RAISES the score)")
print(f"Normal contribution: {nattr.shape}  (importance; sign is NOT 'should bulge')")

ig_sel = sel_test[ig_idx]
ig_pid = pid_test[ig_idx]


# --- Aggregate profiles with a cluster bootstrap over participants ---------
def bootstrap_profile(vals, pids, n_boot=1000, seed=SEED):
    """Mean attribution at each position, with a 95% band from resampling
    PARTICIPANTS (not vases) — same reasoning as Step 8."""
    uniq, inv = np.unique(pids, return_inverse=True)
    sums = np.zeros((len(uniq), vals.shape[1]))
    counts = np.zeros(len(uniq))
    np.add.at(sums, inv, vals)
    np.add.at(counts, inv, 1)
    per_p = sums / counts[:, None]
    r = np.random.default_rng(seed)
    draws = r.integers(0, len(uniq), size=(n_boot, len(uniq)))
    boots = np.empty((n_boot, vals.shape[1]))
    for b in range(n_boot):
        boots[b] = per_p[draws[b]].mean(axis=0)
    return (per_p.mean(axis=0),
            np.percentile(boots, 2.5, axis=0),
            np.percentile(boots, 97.5, axis=0))


prof_all = bootstrap_profile(nsens, ig_pid)
prof_sel = bootstrap_profile(nsens[ig_sel == 1], ig_pid[ig_sel == 1])
prof_rej = bootstrap_profile(nsens[ig_sel == 0], ig_pid[ig_sel == 0])

# mean full contour, for the figures
mean_contour = full_test.mean(axis=0)

np.savez(FIG_DIR / "attribution.npz",
         cam4=cam4_raw, cam2=cam2_raw, nattr=nattr, nsens=nsens, ig_idx=ig_idx,
         prof_all=np.array(prof_all), prof_sel=np.array(prof_sel),
         prof_rej=np.array(prof_rej), half_contour=HALF_CONTOUR)
print(f"Saved attribution arrays to {FIG_DIR / 'attribution.npz'}")


In [ ]:
"""Figure 5 — attribution painted onto individual vases.

Two rows: vases the participant actually SELECTED, and vases they REJECTED.
Within each row the examples are drawn at fixed quantiles of the model's score
rather than at the extremes, so the panel is representative rather than
cherry-picked. Shapes are deduplicated first (see Step 9).

The model scores a half-profile, so the attribution is mirrored back onto the
whole outline — exact, since the halves are the same curve."""

N_COL = 5
QUANTILES = [0.10, 0.30, 0.50, 0.70, 0.90]
ATTR_SOURCE = "cam4"   # "cam4" (decision-faithful) | "cam2" (sharper) | "ig" | "sens"

if ATTR_SOURCE in ("ig", "sens"):
    attr_half = np.zeros((len(X_test), SEQ_LEN), dtype=np.float32)
    attr_half[ig_idx] = nattr if ATTR_SOURCE == "ig" else nsens
    eligible = np.zeros(len(X_test), dtype=bool); eligible[ig_idx] = True
else:
    attr_half = {"cam4": cam4, "cam2": cam2}[ATTR_SOURCE]
    eligible = np.ones(len(X_test), dtype=bool)

attr_plot = to_full(attr_half)        # (n, 250), for drawing on the whole outline

_, first = np.unique(hash_test, return_index=True)
distinct = np.zeros(len(X_test), dtype=bool); distinct[first] = True


def pick(is_selected):
    pool = np.where((sel_test == is_selected) & distinct & eligible)[0]
    pool = pool[np.argsort(test_logits[pool])]
    return [pool[min(int(q * len(pool)), len(pool) - 1)] for q in QUANTILES]


rows = [(1.0, "selected by participant", pick(1.0)),
        (0.0, "rejected by participant", pick(0.0))]

shown_idx = [i for _, _, ids in rows for i in ids]
vmax = float(np.percentile(np.abs(attr_plot[shown_idx]), 99))
norm = plt.Normalize(-vmax, vmax)


def paint(ax, xy, values, lw=3.4):
    """Draw the closed contour as a LineCollection coloured by attribution —
    the natural encoding for a value that varies along a curve."""
    P = np.column_stack([np.append(xy[0], xy[0][0]), np.append(xy[1], xy[1][0])])
    segs = np.stack([P[:-1], P[1:]], axis=1)
    v = np.append(values, values[0])
    lc = LineCollection(segs, cmap=DIVERGING, norm=norm,
                        linewidths=lw, capstyle="round")
    lc.set_array(0.5 * (v[:-1] + v[1:]))
    ax.add_collection(lc)
    return lc


fig, axes = plt.subplots(2, N_COL, figsize=(13.5, 6.4))
pad = 0.3
xl = (full_test[shown_idx][:, 0].min() - pad, full_test[shown_idx][:, 0].max() + pad)
yl = (full_test[shown_idx][:, 1].min() - pad, full_test[shown_idx][:, 1].max() + pad)

for r, (lab, row_title, ids) in enumerate(rows):
    for c, i in enumerate(ids):
        ax = axes[r, c]
        lc = paint(ax, full_test[i], attr_plot[i])
        ax.set_xlim(*xl); ax.set_ylim(*yl); ax.set_aspect("equal"); ax.axis("off")
        ax.set_title(f"p = {test_prob[i]:.3f}", fontsize=8, color=MUTED, pad=4)
        if c == 0:
            ax.text(-0.1, 0.5, row_title, transform=ax.transAxes, rotation=90,
                    va="center", ha="right", fontsize=9,
                    color=BLUE if lab == 1.0 else ORANGE)

cb = fig.colorbar(lc, ax=axes, fraction=0.018, pad=0.015)
cb.ax.tick_params(labelsize=7.5, colors=MUTED, length=2)
cb.outline.set_visible(False)
cb.set_label("← pushes toward rejection      attribution      pushes toward selection →",
             fontsize=8, color=MUTED)

pts_per = SEQ_LEN / {"cam4": cam4_raw, "cam2": cam2_raw}.get(ATTR_SOURCE, cam4_raw).shape[1]
src_label = {
    "cam4": f"CAM at conv4 — {pts_per:.0f} input points per position, interpolated",
    "cam2": f"CAM at conv2 — {pts_per:.0f} input points per position, interpolated",
    "ig":   "Integrated Gradients (contribution, not direction) on the outward normal",
    "sens": "Sensitivity: d(logit)/d(outward displacement)",
}[ATTR_SOURCE]
fig.suptitle(f"Where the model looks — {src_label}", fontsize=11, color=INK, y=0.99)
fig.savefig(FIG_DIR / "05_attribution_examples.png")
plt.show()

print(f"Colour scale: +/-{vmax:.3f} (99th percentile of |attribution| across panels)")
print("Single-vase maps are noisy at this AUC — figure 6 is the stable result.")


In [ ]:
"""Figure 6 — the aggregate: which heights of the vase drive selection.

Panel A folds attribution onto HEIGHT, which is the physically meaningful
question: does this height of the vase want to be wider or narrower? Panel B
shows the same thing as a pressure field. Panel C is the raw profile along the
half-contour from rim to foot — with the redundant half removed from the input,
every position here carries independent signal.

Bootstrap bands resample PARTICIPANTS, since a person's 25 choices are not
independent (same reasoning as Step 8)."""

N_BANDS = 40


def height_profile(attr, contours_full, n_bands=N_BANDS):
    """Fold attribution onto normalised height (0 = foot, 1 = rim).

    Sums rather than averages within a band: points are uniformly spaced in arc
    length, so the SUM is the total influence of that height band."""
    out = np.zeros((len(attr), n_bands), dtype=np.float32)
    for k in range(len(attr)):
        yfull = contours_full[k][1]
        lo, hi = yfull.min(), yfull.max()
        yv = yfull[HALF] if HALF_CONTOUR else yfull
        b = np.clip(((yv - lo) / (hi - lo + 1e-9) * n_bands).astype(int), 0, n_bands - 1)
        np.add.at(out[k], b, attr[k])
    return out


band_centres = (np.arange(N_BANDS) + 0.5) / N_BANDS
h_attr = height_profile(nsens, full_test[ig_idx])
hp_all = bootstrap_profile(h_attr, ig_pid)
hp_sel = bootstrap_profile(h_attr[ig_sel == 1], ig_pid[ig_sel == 1])
hp_rej = bootstrap_profile(h_attr[ig_sel == 0], ig_pid[ig_sel == 0])

fig = plt.figure(figsize=(14.5, 5.0))
gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 0.95, 1.15], wspace=0.3)

# --- Panel A: attribution by height — the interpretable result ------------
ax = fig.add_subplot(gs[0, 0])
for (mean_v, lo, hi), colour, name in [
    (hp_sel, BLUE, "selected vases"),
    (hp_rej, ORANGE, "rejected vases"),
]:
    ax.fill_betweenx(band_centres, lo, hi, color=colour, alpha=0.15, lw=0, zorder=2)
    ax.plot(mean_v, band_centres, color=colour, lw=2, zorder=3, label=name)

ax.axvline(0, color=MUTED, lw=0.9, alpha=0.7, zorder=1)
ax.annotate("wants to\npinch in", xy=(0.03, 0.02), xycoords="axes fraction",
            fontsize=7.5, color=MUTED, ha="left", va="bottom")
ax.annotate("wants to\nbulge out", xy=(0.97, 0.02), xycoords="axes fraction",
            fontsize=7.5, color=MUTED, ha="right", va="bottom")
style(ax, "Selection pressure by height",
      "d(logit) / d(outward displacement)", "height up the vase (0 = foot, 1 = rim)",
      grid="both")
ax.set_ylim(0, 1)
ax.legend(frameon=False, fontsize=8.5, labelcolor=MUTED, loc="upper right")

# --- Panel B: the same thing as a pressure field on the mean vase ---------
ax = fig.add_subplot(gs[0, 1])
N = outward_normals(mean_contour)
yv = mean_contour[1]
band_of_pt = np.clip(((yv - yv.min()) / (np.ptp(yv) + 1e-9) * N_BANDS).astype(int),
                     0, N_BANDS - 1)
vals = hp_all[0][band_of_pt]

vmax_q = float(np.percentile(np.abs(vals), 98))
norm = plt.Normalize(-vmax_q, vmax_q)
# clip arrow LENGTH so one outlier band cannot blow up the panel; colour still
# carries the true value
span = float(np.ptp(mean_contour[0]))
arrow_len = np.clip(vals, -vmax_q, vmax_q) / (vmax_q + 1e-12) * (0.22 * span)

P = np.column_stack([np.append(mean_contour[0], mean_contour[0][0]),
                     np.append(mean_contour[1], mean_contour[1][0])])
ax.plot(P[:, 0], P[:, 1], color=GRAY_MARK, lw=1.6, zorder=2)

step = 5
sel = np.arange(0, FULL_LEN, step)
q = ax.quiver(mean_contour[0][sel], mean_contour[1][sel],
              N[sel, 0] * arrow_len[sel], N[sel, 1] * arrow_len[sel],
              vals[sel], cmap=DIVERGING, norm=norm,
              angles="xy", scale_units="xy", scale=1,
              width=0.008, headwidth=3.5, headlength=4, zorder=4)
ax.set_aspect("equal"); ax.axis("off"); ax.margins(0.28)
ax.set_title("Pressure field on the average vase", fontsize=10, color=INK, pad=10)
ax.annotate("outward = bulging here raises the score",
            xy=(0.5, -0.02), xycoords="axes fraction", ha="center", va="top",
            fontsize=7.5, color=MUTED)
cb = fig.colorbar(q, ax=ax, fraction=0.045, pad=0.04)
cb.ax.tick_params(labelsize=7.5, colors=MUTED, length=2)
cb.outline.set_visible(False)

# --- Panel C: along the half-profile, rim -> foot --------------------------
ax = fig.add_subplot(gs[0, 2])
pos = np.arange(SEQ_LEN)
for (mean_v, lo, hi), colour, name in [(prof_sel, BLUE, "selected"),
                                       (prof_rej, ORANGE, "rejected")]:
    ax.fill_between(pos, lo, hi, color=colour, alpha=0.15, lw=0, zorder=2)
    ax.plot(pos, mean_v, color=colour, lw=1.8, zorder=3, label=name)
ax.axhline(0, color=MUTED, lw=0.9, alpha=0.7, zorder=1)
style(ax, "Along the half-profile", "position (0 = rim, %d = foot)" % (SEQ_LEN - 1),
      "d(logit) / d(outward displacement)")
ax.set_xlim(0, SEQ_LEN - 1)
ax.legend(frameon=False, fontsize=8.5, labelcolor=MUTED, loc="lower right")

fig.suptitle(f"Aggregate sensitivity — {len(ig_idx):,} test vases, "
             f"half-profile input ({SEQ_LEN} points), "
             "participant-level bootstrap bands",
             fontsize=11, color=INK, y=1.01)
fig.savefig(FIG_DIR / "06_attribution_aggregate.png")
plt.show()

# --- Readout ---------------------------------------------------------------
sig = (hp_all[1] > 0) | (hp_all[2] < 0)
print(f"Height bands whose 95% band excludes zero: {sig.sum()} of {N_BANDS}\n")
print(f"{'height':>8}{'attribution':>14}   direction")
for b in np.argsort(np.abs(hp_all[0]))[::-1][:6]:
    d = "bulge OUT" if hp_all[0][b] > 0 else "pinch IN"
    print(f"{band_centres[b]:>8.2f}{hp_all[0][b]:>+14.4f}   {d}{' *' if sig[b] else ''}")
print("\n* = 95% bootstrap band excludes zero")


## Step 12 — Validation: does the attribution describe *people*, or just the model?

An attribution map explains the **model**. Whether it also describes what participants prefer is a separate question that has to be tested, not assumed — especially at AUC 0.60, where the model's gradients are weak.

The test fits a within-pair logistic regression **directly to the choices**: absolute half-width in each of 20 height bands plus overall height, on the difference between the two vases of a trial, antisymmetrised so presentation order carries no information. Its coefficients are partial effects of shape on choice *by construction*. If the CNN's sensitivity map describes the same thing, the two should agree band by band.

They largely do not. Run the cell for the current numbers; on the run this notebook was written against, the correlation across height bands was **r ≈ +0.07** with sign agreement at **60%** against a 50% chance baseline.

**So figure 6 should not be read as "people want vases narrower at height X".** It is a correct description of the model's local gradients and nothing stronger.

One caveat cuts the other way: displacing a contour point along its outward normal changes that point's *height* as well as its radius, except where the profile is vertical. "Outward-normal sensitivity" and "partial effect of width at height h" are therefore related but not identical, so some disagreement is expected even if both are computed correctly. The comparison bounds the claim; it does not by itself prove the attribution wrong.

A useful side result: the linear width-profile model reaches ~0.617 in-sample against the CNN's 0.648 held-out. Most of the available signal is already linear in the width profile, which is worth knowing before investing in a more elaborate architecture.

In [ ]:
"""Step 12 — Does the attribution agree with the choices themselves?

An attribution map explains the MODEL. Whether it also describes what people
prefer is a separate question, and it has to be tested rather than assumed.

The test: fit a within-pair logistic regression directly to the choices, using
absolute half-width in each of 20 height bands plus overall height, on the
difference between the two vases of each trial (antisymmetrised so presentation
order carries no information). Its coefficients ARE partial effects of shape on
choice, by construction. If the CNN's sensitivity map describes the same thing,
the two should agree band by band.

Caveat on the comparison itself: displacing a contour point along its outward
normal changes the point's HEIGHT as well as its radius, except where the
profile is vertical. So "outward-normal sensitivity" and "partial effect of
width at height h" are related but not identical quantities, and some
disagreement is expected even if both are correct."""

from sklearn.linear_model import LogisticRegression

NB_VAL = 20
edges = np.linspace(0, 1, NB_VAL + 1)
centres = 0.5 * (edges[:-1] + edges[1:])

W = np.zeros((len(X_full), NB_VAL))
Hgt = np.zeros(len(X_full))
for k in range(len(X_full)):
    yv = X_full[k][1]
    Hgt[k] = np.ptp(yv)
    b = np.clip(np.digitize((yv - yv.min()) / (Hgt[k] + 1e-9), edges[1:-1]), 0, NB_VAL - 1)
    for j in range(NB_VAL):
        m = b == j
        if m.any():
            W[k, j] = np.abs(X_full[k][0][m]).max()

dW = W.reshape(-1, 2, NB_VAL)[:, 0, :] - W.reshape(-1, 2, NB_VAL)[:, 1, :]
dH = (Hgt.reshape(-1, 2)[:, 0] - Hgt.reshape(-1, 2)[:, 1])[:, None]
Xd = np.hstack([dW, dH])
yd = (y.reshape(-1, 2)[:, 0] == 1).astype(int)
Xd = np.vstack([Xd, -Xd]); yd = np.concatenate([yd, 1 - yd])   # antisymmetrise

lr = LogisticRegression(fit_intercept=False, max_iter=2000).fit(Xd, yd)
lr_coef = lr.coef_[0][:NB_VAL]
print(f"Linear width-profile model, in-sample pairwise accuracy: {lr.score(Xd, yd):.4f}")
print(f"VaseCNN, held-out pairwise accuracy:                     {te_pair:.4f}")
print(f"-> most of the available signal is already linear in the width profile.\n")

# CNN sensitivity folded onto the same 20 bands
cnn_h = np.zeros((len(ig_idx), NB_VAL))
for i, k in enumerate(ig_idx):
    yv = full_test[k][1]
    f = (yv[HALF] - yv.min()) / (np.ptp(yv) + 1e-9) if HALF_CONTOUR else \
        (yv - yv.min()) / (np.ptp(yv) + 1e-9)
    b = np.clip((f * NB_VAL).astype(int), 0, NB_VAL - 1)
    np.add.at(cnn_h[i], b, nsens[i])
cnn_h = cnn_h.mean(axis=0)

agree = np.sign(cnn_h) == np.sign(lr_coef)
r = np.corrcoef(cnn_h, lr_coef)[0, 1]
print(f"{'height':>7}{'CNN sensitivity':>18}{'choice-fit coef':>18}{'same sign':>11}")
for i in range(NB_VAL):
    print(f"{centres[i]:>7.2f}{cnn_h[i]:>+18.4f}{lr_coef[i]:>+18.3f}"
          f"{'  yes' if agree[i] else '   NO':>11}")
print(f"\ncorrelation across height bands: r = {r:+.3f}")
print(f"sign agreement: {agree.mean():.0%} of {NB_VAL} bands (50% = chance)")
print(f"""
VERDICT
-------
The attribution maps in figures 5 and 6 are correctly computed and they
faithfully describe THIS MODEL. They are not, on this evidence, a validated
description of human preference: the model's per-height sensitivity does not
track the partial effects a regression fitted directly to the choices finds
(r = {r:+.3f}, sign agreement {agree.mean():.0%}).

Do NOT read a claim like "people want vases narrower at height X" off figure 6.
For statements about preference, the coefficients above are the better
instrument — they are fitted to the choices, their signs are partial effects by
construction, and the linear model is nearly as accurate as the CNN.

Use the CNN for prediction and for locating WHERE the signal is (|sensitivity|
is more robust than its sign); use the regression for direction.""")
